Python notebook to convert gliner saved results to Label studio annotations.

## Data and Libraries

In [1]:
import json
import uuid
import os
import re

import json
from datetime import datetime, timezone

from collections import defaultdict, Counter

import multiset_multicover as mm

from dataset_processing import CLIRENER_LABELS_V1, BIODIVNER_LABELS, CLIMATEIE_LABELS, IBMCCNER_LABELS

/home/p0l3/miniforge3/envs/clirener_finetune/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Helper Functions

In [2]:
def extract_paper_id(filename):
    match = re.match(r"\d+_(\d+)_", filename)
    return match.group(1) if match else "unknown"

def convert_to_labelstudio_format(data, paper_id, model_version="gliner-community/gliner_medium-v2.5"):
    if not isinstance(data, list):
        raise TypeError("Expected data to be a list of dictionaries, but got: {}".format(type(data)))
    
    labelstudio_data = []
    
    for item in data:
        if not isinstance(item, dict) or "entities" not in item:
            continue
        
        predictions = {
            "model_version": model_version,
            "score": 0.5,  # Default score for the overall prediction
            "result": []
        }
        
        for entity in item.get("entities", []):
            if not isinstance(entity, dict):
                continue
            
            predictions["result"].append({
                "id": str(uuid.uuid4())[:8],
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    "start": entity.get("start", 0),
                    "end": entity.get("end", 0),
                    "score": entity.get("score", 0.0),
                    "text": entity.get("text", ""),
                    "labels": [entity.get("label", "Unknown")]
                }
            })
        
        item["paper_id"] = paper_id
        labelstudio_data.append({"data": item, "predictions": [predictions]})
    
    return labelstudio_data

def process_directory(input_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for filename in os.listdir(input_dir):
        if filename.endswith(".json"):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)
            
            paper_id = extract_paper_id(filename)
            
            with open(input_path, "r", encoding="utf-8") as infile:
                try:
                    data = json.load(infile)
                except json.JSONDecodeError:
                    print(f"Skipping {filename}: Invalid JSON format.")
                    continue
            
            try:
                transformed_data = convert_to_labelstudio_format(data, paper_id)
            except TypeError as e:
                print(f"Skipping {filename}: {e}")
                continue
            
            with open(output_path, "w", encoding="utf-8") as outfile:
                json.dump(transformed_data, outfile, indent=4)
                
def format_for_label_studio(alldata, project_id=20, start_task_id=0, start_prediction_id=0):
    """
    Wraps pre-formatted data and predictions into the full Label Studio task format.
    Handles a list of lists (papers) as input.

    Args:
        alldata (list of lists): A list of papers, where each paper is a list of sentence dicts.
        project_id (int): The ID of the Label Studio project these tasks belong to.
        start_task_id (int): The starting ID for the main tasks.
        start_prediction_id (int): The starting ID for the prediction objects.

    Returns:
        list: A list of tasks formatted for Label Studio import.
    """
    formatted_tasks = []
    
    # Initialize a global counter for unique IDs across all papers
    task_counter = 0

    for paper in alldata:
        for item in paper:
            # Use the global counter to ensure unique IDs
            task_id = start_task_id + task_counter
            prediction_id = start_prediction_id + task_counter

            # Get the current time in the required ISO format with 'Z'
            now_utc = datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

            if not item.get('predictions'):
                continue
                
            source_prediction = item['predictions'][0]

            # Build the new, fully-structured task dictionary
            new_task = {
                "id": task_id,
                "data": item['data'],
                "annotations": [],
                "predictions": [
                    {
                        "id": prediction_id,
                        "result": source_prediction['result'],
                        "model_version": source_prediction.get('model_version'),
                        "created_ago": "0 minutes",
                        "score": source_prediction.get('score'),
                        "cluster": None,
                        "neighbors": None,
                        "mislabeling": 0,
                        "created_at": now_utc,
                        "updated_at": now_utc,
                        "model": None,
                        "model_run": None,
                        "task": task_id,
                        "project": project_id
                    }
                ]
            }
            formatted_tasks.append(new_task)
            
            # Increment the global counter for the next task
            task_counter += 1

    return formatted_tasks

def merge_annotations(formatted_tasks, annots_data):
    """
    Merges annotations from a Label Studio export into a list of formatted tasks.

    Args:
        formatted_tasks (list): The output from format_for_label_studio.
        annots_data (list): The original data from the Label Studio export.

    Returns:
        list: The formatted_tasks list with annotations added where they match.
    """
    # Create a fast lookup map from the annotations data
    annotations_map = {
        (task['data']['paper_id'], task['data']['sentence_id']): task.get('annotations', [])
        for task in annots_data
        if 'data' in task and 'paper_id' in task['data'] and 'sentence_id' in task['data']
    }

    # Iterate through the tasks and add the annotations
    for task in formatted_tasks:
        lookup_key = (task['data']['paper_id'], task['data']['sentence_id'])
        
        # If a matching annotation is found in the map, update the task
        if lookup_key in annotations_map:
            task['annotations'] = annotations_map[lookup_key]
            
    return formatted_tasks

label_remap_rules_std = {
        "Artefact": "Intellectual Artefact",
        "Astronomical Object": "Location"
    }

label_remap_rules_all2other = {
    "Artefact": "Other",
    "Astronomical Object": "Other",
    "Body Part": "Other",
    "Body of Water": "Other",
    "Chemical": "Other",
    "Disease": "Other",
    "Ecosystem": "Other",
    "Energy Source": "Other",
    "Field of Study": "Other",
    "Geographical Feature": "Other",
    "Location": "Other",
    "Mathematical Expression": "Other",
    "Measuring Device": "Other",
    "Meteorological Phenomenon": "Other",
    "Method": "Other",
    "Natural Disaster": "Other",
    "Natural Phenomenon": "Other",
    "Organism": "Other",
    "Organization": "Other",
    "Person": "Other",
    "Physical Phenomenon": "Other",
    "Quantity": "Other",
    "Satellite": "Other",
    "System": "Other",
    "Time Period": "Other"
}

def update_entity_labels(file_path, label_remap_rules = label_remap_rules_std):
    """
    Loads a Label Studio JSON file, remaps specific entity labels,
    and saves the file back.

    Args:
        file_path (str): The path to the JSON file to be updated.
    """
    # Define the remapping rules for old labels to new labels
    
    print(f"Loading data from '{file_path}'...")
    
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return
    except json.JSONDecodeError:
        print(f"Error: The file '{file_path}' is not a valid JSON file.")
        return

    update_count = 0

    # This helper function will perform the update on a list of results
    def process_results(results):
        nonlocal update_count
        for result_item in results:
            if result_item.get('type') == 'labels' and 'value' in result_item and 'labels' in result_item['value']:
                # The 'labels' field is a list, e.g., ["Artefact"]
                # We update it in place
                original_labels = result_item['value']['labels']
                updated_labels = []
                for label in original_labels:
                    # If the label is in our rules, use the new value, otherwise keep the old one
                    if label in label_remap_rules:
                        updated_labels.append(label_remap_rules[label])
                        update_count += 1
                    else:
                        updated_labels.append(label)
                result_item['value']['labels'] = updated_labels

    # Iterate through every task in the dataset
    for task in data:
        # 1. Update labels within the 'annotations' section
        if 'annotations' in task:
            for annotation in task['annotations']:
                if 'result' in annotation:
                    process_results(annotation['result'])
        
        # 2. Update labels within the 'predictions' section
        if 'predictions' in task:
            for prediction in task['predictions']:
                if 'result' in prediction:
                    process_results(prediction['result'])

    if update_count > 0:
        print(f"Successfully updated {update_count} label occurrences.")
        print(f"Saving updated data back to '{file_path}'...")
        
        # Save the modified data back to the same file with pretty printing
        with open(file_path, 'w') as f:
            json.dump(data, f, indent=2)
            
        print("File saved successfully.")
    else:
        print("No labels matching the remapping rules were found. The file was not changed.")

def create_shorthand_string(entity_types):
    """Creates a single string of entity type shorthands from a list of full names."""
    shorthands = []
    for entity in entity_types:
        words = entity.split(' ')
        parts = [word[:3].capitalize() for word in words]
        shorthand = ''.join(parts)
        shorthands.append(shorthand)
    return '_'.join(shorthands)

def remove_annots(labelstudio_data):
    """
    Removes all annotations from a list of Label Studio tasks.

    This function iterates through each task and replaces the value of the
    'annotations' key with an empty list. It modifies the data in-place.

    Args:
        labelstudio_data (list): A list of task dictionaries from Label Studio.

    Returns:
        list: The same list of tasks with the annotations cleared.
    """
    for task in labelstudio_data:
        if 'annotations' in task:
            task['annotations'] = []
    return labelstudio_data

def set_all_labels_to_other(data):
    """
    Traverses the nested prediction structure and sets all 
    entity labels to ["Other"].
    """
    for task in data:
        # Get predictions list (safe check if key exists)
        predictions = task.get("predictions", [])
        
        for prediction in predictions:
            # Get results list
            results = prediction.get("result", [])
            
            for entity_prediction in results:
                # Get the 'value' dictionary
                value_data = entity_prediction.get("value")
                
                # Check if value_data exists to avoid errors
                if value_data:
                    # Overwrite the labels
                    value_data["labels"] = ["Other"]
    
    return data

def filter_json_data(source_file_path, filter_file_path, output_file_path, annots=False, default_labels=True):
    """
    Filters data from source_file based on keys present in filter_file.
    
    Args:
        source_file_path (str): Path to the larger dataset.
        filter_file_path (str): Path to the smaller dataset used as a filter.
        output_file_path (str): Path where the result JSON will be saved.
        annots (boolean): Save previous annotation. Default: False
        default_labels (boolean): Set all labels to "Other". Default: True
    """
    
    # 1. Load the data
    try:
        with open(source_file_path, "r") as f:
            source_data = json.load(f)
        with open(filter_file_path, "r") as f:
            filter_data = json.load(f)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return

    # 2. Create a set of keys from the 'filter' file for fast lookup
    # We use a set() because looking up items in a set is much faster than a list
    target_keys = set()
    
    for task in filter_data:
        task_data = task.get("data", {})
        paper_id = task_data.get("paper_id")
        sentence_id = task_data.get("sentence_id")
        
        if paper_id is not None and sentence_id is not None:
            # Create the compound key
            compound_id = f"{paper_id}-{sentence_id}"
            target_keys.add(compound_id)

    print(f"Loaded {len(target_keys)} unique keys to filter by.")

    # 3. Filter the source data
    filtered_results = []
    
    for task in source_data:
        task_data = task.get("data", {})
        paper_id = task_data.get("paper_id")
        sentence_id = task_data.get("sentence_id")
        
        # Create key for current item
        current_compound_id = f"{paper_id}-{sentence_id}"
        
        # Check if this key exists in our target set
        if current_compound_id in target_keys:
            filtered_results.append(task)

    # 4. Save the output
    # Ensure directory exists
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    
    if not annots:
        filtered_results = remove_annots(filtered_results)

    if default_labels:
        filtered_results = set_all_labels_to_other(filtered_results)

    with open(output_file_path, "w") as f:
        json.dump(filtered_results, f, indent=4)

    print(f"Filtering complete.")
    print(f"Source count: {len(source_data)}")
    print(f"Filtered count: {len(filtered_results)}")
    print(f"Data contains previous annotations: {annots}")
    print(f"Data has all labels set as \"Other\": {default_labels}")
    print(f"Saved to: {output_file_path}")

## GLiNER to Label Studio

In [4]:
# Example usage (Change accordingly)
input_directory = "C:\\Users\\ANDRIJA_RAD\\CWED4ETA\\CWED4ETA\\RESULTS\\GLINER_TESTS_FULL_27_50S" # For boundary analysis on review 14.7.2026.
# "RESULTS/BioDivNER/" # 3.12.2025.
# "RESULTS/gliner_medium_v2_5_CliReNER_v_1_1_28_SILVER_checkpoint-final_V1_PSS/" # 26.11.2025.
# "RESULTS/GLINER_MEDIUM_V21_CLIRENER_2025_11_11_12_33_f873727e_V1_PSS/"
output_directory = "RESULTS/GLiNER_nonfinetuned/" # 14.7.2026.
# "RESULTS/BioDivNER_LS/" # 3.12.2025.
# "RESULTS/gliner_medium_v2_5_CliReNER_v_1_1_28_SILVER_checkpoint-final_V1_PSS_LS/"
# "RESULTS/GLINER_MEDIUM_V21_CLIRENER_2025_11_11_12_33_f873727e_V1_PSS_LS/"
process_directory(input_directory, output_directory)


## Label Studio Merge

First, we connect all the GLiNER predictions into a single list of jsons.

In [5]:
input_directory = "RESULTS/GLiNER_nonfinetuned/" # 14.7.2026.
# "RESULTS/BioDivNER_LS/" # 3.12.2025.
# "RESULTS/gliner_medium_v2_5_CliReNER_v_1_1_28_SILVER_checkpoint-final_V1_PSS_LS/" # 26.11.2025.
# "RESULTS/GLINER_MEDIUM_V21_CLIRENER_2025_11_11_12_33_f873727e_V1_PSS_LS/" 
# "RESULTS/GLINER_MEDIUM_V25_V1_PSS_LS" # "RESULTS/GLINER_MEDIUM_V21_CLIRENER_2025_11_11_12_33_f873727e_V1_PSS_LS/"
output_file_path = "RESULTS/GLiNER_nonfinetuned/gliner_nonfinetuned_full_ls.json" # 14.7.2026.
# "RESULTS/BioDivNER_LS/biodivner_full_ls.json" # 3.12.2025.
# "RESULTS/gliner_medium_v2_5_CliReNER_v_1_1_28_SILVER_checkpoint-final_v1_pss_ls_singlefile.json"
# "RESULTS/gliner_medium_v21_clirener_2025_11_11_12_33_f873727e_v1_pss_ls_singlefile.json"
# "RESULTS/gliner_medium_v25_v1_pss_ls_singlefile.json" # "RESULTS/gliner_medium_v25_clirener_2025_11_11_12_33_f873727e_v1_pss_ls_singlefile.json"

output_file = []
for filename in os.listdir(input_directory):
    if filename.endswith(".json"):
        input_path = os.path.join(input_directory, filename)
             
        with open(input_path, "r", encoding="utf-8") as infile:
            try:
                data = json.load(infile)
            except json.JSONDecodeError:
                print(f"Skipping {filename}: Invalid JSON format.")
                continue
        
        output_file.append(data)
        
with open(output_file_path, "w", encoding="utf-8") as outfile:
    json.dump(output_file, outfile, indent=4)

Next, we want to update the data with arbitrary annotations:

In [7]:
annots_DIR = "RESULTS/GLiNER_nonfinetuned/gliner_nonfinetuned_full_ls.json" # 14.7.2026.
# "DATA/LABEL_STUDIO/project-30-at-2025-11-14-12-19-2a7464a5.json" # "DATA/LABEL_STUDIO/project-28-at-2025-11-11-12-33-f873727e.json"
alldata_DIR = "RESULTS/GLiNER_nonfinetuned/gliner_nonfinetuned_full_ls.json" # 14.7.2026.
# "RESULTS/gliner_medium_v2_5_CliReNER_v_1_1_28_SILVER_checkpoint-final_v1_pss_ls_singlefile.json" # 26.11.2025.
# "RESULTS/gliner_medium_v21_clirener_2025_11_11_12_33_f873727e_v1_pss_ls_singlefile.json"
# "RESULTS/gliner_medium_v25_v1_pss_ls_singlefile.json" # "RESULTS/gliner_medium_v21_clirener_2025_11_11_12_33_f873727e_v1_pss_ls_singlefile.json"


with open(annots_DIR, "r") as f:
    annots_data = json.load(f)

with open(alldata_DIR, "r") as f:
    alldata_data = json.load(f)

First, let's convert GLiNER annotated data to simulate Label Studio format:

In [8]:
# 2. Perform the transformation
final_data = format_for_label_studio(alldata_data)

# 3. Print the result in a readable JSON format
# print(json.dumps(final_data, indent=2))

# 4. (Optional) Save the result to a file for import
alldata_ls_DIR = alldata_DIR.split(".json")[0]+"_lsformat.json"
with open(alldata_ls_DIR, 'w') as f:
    json.dump(final_data, f, indent=2)

print(f"\nTransformation complete. See the output above or in '{alldata_ls_DIR}'.")


Transformation complete. See the output above or in 'RESULTS/GLiNER_nonfinetuned/gliner_nonfinetuned_full_ls_lsformat.json'.


In [9]:
alldata_ls_DIR = 'RESULTS/GLiNER_nonfinetuned/gliner_nonfinetuned_full_ls_lsformat.json' # 14.7.2026.
# 'RESULTS/gliner_medium_v2_5_CliReNER_v_1_1_28_SILVER_checkpoint-final_v1_pss_ls_singlefile_lsformat.json' # 26.11.2025.
# 'RESULTS/gliner_medium_v21_clirener_2025_11_11_12_33_f873727e_v1_pss_ls_singlefile_lsformat.json'
# "RESULTS/gliner_medium_v25_v1_pss_ls_singlefile_lsformat.json" 
# "RESULTS/gliner_medium_v21_clirener_2025_11_11_12_33_f873727e_v1_pss_ls_singlefile_lsformat.json"
with open(alldata_ls_DIR, "r") as f:
    alldata_ls = json.load(f)

# Step 2: Merge the existing annotations into the newly formatted tasks
final_merged_data = merge_annotations(alldata_ls, annots_data)

Save data ready for loading:

In [10]:
final_merged_data_DIR = 'RESULTS/GLiNER_nonfinetuned/gliner_nonfinetuned_full_anotserafy.json' # 14.7.2026.
# 'RESULTS/gliner_m_v2_5_CliReNER_v_1_1_28_SILVER_v1_261125_anotsready.json' # 26.11.2025.
# 'RESULTS/gliner_medium_v21_clirener_2025_11_11_12_33_f873727e_anotsready.json'
# "RESULTS/gliner_med_v25_v1_anotsready.json"
# "RESULTS/gliner_med_v21_clirener_2025_11_11_12_33_f873727e_v1_anotsready.json"
with open(final_merged_data_DIR, 'w') as f:
    json.dump(final_merged_data, f, indent=2)

Note that from v0 to v1 there have been few changes, adding few new entity types: "Asset, Physical Artefact and Policy" and two remapings: "Astronomical Object"->"Location" ("Astronomical Object" is deleted) and "Artefact"->"Intellectual Artefact". Thus, annotations should have these updated, especially "Artefact" and "Astronomical Object":

In [11]:
final_merged_data_DIR = 'RESULTS/GLiNER_nonfinetuned/gliner_nonfinetuned_full_anotserafy.json' # 14.7.2026.
# 'RESULTS/gliner_m_v2_5_CliReNER_v_1_1_28_SILVER_v1_261125_anotsready.json' # 26.11.2025.
# 'RESULTS/gliner_medium_v21_clirener_2025_11_11_12_33_f873727e_anotsready.json'
# "RESULTS/gliner_med_v25_v1_anotsready.json" # "RESULTS/gliner_med_v21_clirener_2025_11_11_12_33_f873727e_v1_anotsready.json"
# Run the function to update the labels
update_entity_labels(final_merged_data_DIR)

Loading data from 'RESULTS/GLiNER_nonfinetuned/gliner_nonfinetuned_full_anotserafy.json'...
Successfully updated 2396 label occurrences.
Saving updated data back to 'RESULTS/GLiNER_nonfinetuned/gliner_nonfinetuned_full_anotserafy.json'...
File saved successfully.


Finally, this (gliner_med_v25_v1_anotsready) can be loaded into the Label Studio and used further for annotation.

## Annotator Data Creator

### Create Multiset for all 27 classes (-"Other")

First let's get the golden set:

In [4]:
def select_and_split_samples_with_msc_api(file_path, n_sample, entity_types_to_tag, golden_filename, training_filename, look="annotations"):
    """
    Selects sentences using the multiset-multicover API based on either annotations or predictions,
    and splits the data into a golden set and a training set.

    Args:
        file_path (str): Path to the input JSON file.
        n_sample (int): The target number of examples for each entity type.
        entity_types_to_tag (list): A list of entity type strings to focus on.
        golden_filename (str): Path to save the selected "golden" data.
        training_filename (str): Path to save the remaining "training" data.
        look (str): Determines the source of labels. Options: "annotations" (default) or "predictions".

    Returns:
        tuple: A tuple containing the list of selected tasks and remaining tasks.
    """
    print(f"--- Starting Sample Selection and Splitting ({look}) using multiset-multicover API ---")

    if look not in ["annotations", "predictions"]:
        raise ValueError("The 'look' parameter must be either 'annotations' or 'predictions'.")

    # 1. Load data and filter for tasks containing the specific source data
    try:
        with open(file_path, 'r') as f:
            all_tasks = json.load(f)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return [], []

    # Filter tasks that actually contain the requested key (annotations or predictions) and have results
    candidate_tasks = [
        task for task in all_tasks
        if task.get(look) and task[look][0].get('result')
    ]
    
    print(f"Found {len(candidate_tasks)} sentences with existing {look} to select from.")
    if not candidate_tasks:
        print(f"No sentences with {look} found. Exiting.")
        return [], []

    # 2. Create mappings and initialize the solver
    entity_to_idx = {entity: i for i, entity in enumerate(entity_types_to_tag)}
    num_entity_types = len(entity_types_to_tag)
    gci = mm.GreedyCoverInstance(num_entity_types)
    
    task_map = []

    print("Preprocessing data for the solver...")
    for task in candidate_tasks:
        # Access the list of results based on the 'look' parameter
        result_list = task[look][0]['result']
        
        entity_counts_in_task = Counter(
            item['value']['labels'][0]
            for item in result_list
            if 'value' in item and 'labels' in item['value'] and item['value']['labels'][0] in entity_types_to_tag
        )
        
        if entity_counts_in_task:
            elements = [entity_to_idx[entity] for entity, count in entity_counts_in_task.items()]
            multiplicities = [count for entity, count in entity_counts_in_task.items()]

            gci.add_multiset(elements, multiplicity=multiplicities)
            task_map.append(task)

    if not task_map:
        print("No tasks found containing the specified entity types.")
        # Save all candidate tasks to the training file as none are selected for golden set
        print(f"\nSaving 0 selected sentences to '{golden_filename}'...")
        with open(golden_filename, 'w') as f:
            json.dump([], f, indent=2)

        print(f"Saving all {len(candidate_tasks)} candidate sentences to '{training_filename}'...")
        with open(training_filename, 'w') as f:
            json.dump(candidate_tasks, f, indent=2)
        print("\nDone.")
        return [], candidate_tasks

    # 3. Define coverage requirements
    requirements = [n_sample] * num_entity_types
    print(f"\nSeeking to cover {num_entity_types} entity types with {n_sample} examples each.")

    # 4. Run the solver
    print("Running the multiset-multicover solver...")
    selected_indices_in_map = gci.cover(requirements)
    print(f"Solver selected {len(selected_indices_in_map)} sentences.")

    # 5. Retrieve selected tasks and identify remaining tasks
    selected_tasks = [task_map[i] for i in selected_indices_in_map]
    selected_task_ids = {id(task) for task in selected_tasks}
    
    # Filter from 'candidate_tasks' to ensure remaining tasks also have the required data (annotations/predictions)
    remaining_tasks = [
        task for task in candidate_tasks if id(task) not in selected_task_ids
    ]

    # 6. Save the selected and remaining data to separate files
    print(f"\nSaving {len(selected_tasks)} selected sentences to '{golden_filename}'...")
    with open(golden_filename, 'w') as f:
        json.dump(selected_tasks, f, indent=2)

    print(f"Saving {len(remaining_tasks)} remaining sentences to '{training_filename}'...")
    with open(training_filename, 'w') as f:
        json.dump(remaining_tasks, f, indent=2)

    # 7. Print selection summary
    print("\n--- Selection Summary ---")
    final_counts = Counter()
    for task in selected_tasks:
        # Count based on the 'look' parameter
        for item in task[look][0]['result']:
            try:
                label = item['value']['labels'][0]
                if label in entity_types_to_tag:
                    final_counts[label] += 1
            except (KeyError, IndexError):
                continue

    print(f"Final counts per entity type in the golden set (based on {look}):")
    for entity in entity_types_to_tag:
        count = final_counts.get(entity, 0)
        print(f"  - {entity}: {count}/{n_sample} examples")

    print("\nDone.")
    return selected_tasks, remaining_tasks

# --- EXAMPLE USAGE ---

# Define your parameters WATCH OUT FOR ENTITY TYPES TO TAG
FILE_PATH = "RESULTS/gliner_m_v2_5_CliReNER_v_1_1_28_SILVER_v1_261125_anotsready.json" # 14.7.2026.
"RESULTS/FT_ANNOTS/gliner_med_v25_v1_anotsready_OLD.json" # 14.7.2026.
"RESULTS/IBMCCNER/ibmccner_full.json"
"RESULTS/CLIMATEIE/climateie_full.json"
"RESULTS/CLIMATEIE/climateie_full.json"
"RESULTS/BioDivNER/biodivner_full.json" # 3.12.2025.
# "RESULTS/FT_ANNOTS/gliner_med_v25_v1_anotsready_OLD.json"
N_SAMPLES_PER_TYPE = 50

ENTITY_TYPES_TO_TAG = list(CLIRENER_LABELS_V1)
# list(IBMCCNER_LABELS)
# list(BIODIVNER_LABELS) # 3.12.2025.
# list(CLIRENER_LABELS_V1)

# For CLIRENER
if "Other" in ENTITY_TYPES_TO_TAG:
    ENTITY_TYPES_TO_TAG.remove("Other")
    
# # For CLIMATEIE
# if "event" in ENTITY_TYPES_TO_TAG:
#     ENTITY_TYPES_TO_TAG.remove("event")
# if "ocean circulation" in ENTITY_TYPES_TO_TAG:
#     ENTITY_TYPES_TO_TAG.remove("ocean circulation")
# Generate the filenames from the entity types
filename_shorthand = create_shorthand_string(ENTITY_TYPES_TO_TAG)
output_dir = "RESULTS/annots_v2_ft_silver_all_labels" # 14.7.2026.
# "RESULTS/OTHER_DATASETS_ANNOTATIONS"
# "RESULTS/FT_ANNOTS"
os.makedirs(output_dir, exist_ok=True)

golden_output_filename = os.path.join(output_dir, f"golden_{N_SAMPLES_PER_TYPE}_{filename_shorthand}.json")
training_output_filename = os.path.join(output_dir, f"training_{N_SAMPLES_PER_TYPE}_{filename_shorthand}.json")


selected_data, remaining_data = select_and_split_samples_with_msc_api(
    file_path=FILE_PATH,
    n_sample=N_SAMPLES_PER_TYPE,
    entity_types_to_tag=ENTITY_TYPES_TO_TAG,
    golden_filename=golden_output_filename,
    training_filename=training_output_filename,
    look="predictions"
)

--- Starting Sample Selection and Splitting (predictions) using multiset-multicover API ---
Found 11949 sentences with existing predictions to select from.
Preprocessing data for the solver...

Seeking to cover 27 entity types with 50 examples each.
Running the multiset-multicover solver...
Solver selected 148 sentences.

Saving 148 selected sentences to 'RESULTS/annots_v2_ft_silver_all_labels/golden_50_NatPhe_Org_Eco_MetPhe_PhyArt_Ass_Sat_NatDis_Per_Sys_BodOfWat_Dis_Met_Qua_BodPar_MeaDev_PhyPhe_GeoFea_Loc_TimPer_EneSou_Che_Pol_IntArt_FieOfStu_Org_MatExp.json'...
Saving 11801 remaining sentences to 'RESULTS/annots_v2_ft_silver_all_labels/training_50_NatPhe_Org_Eco_MetPhe_PhyArt_Ass_Sat_NatDis_Per_Sys_BodOfWat_Dis_Met_Qua_BodPar_MeaDev_PhyPhe_GeoFea_Loc_TimPer_EneSou_Che_Pol_IntArt_FieOfStu_Org_MatExp.json'...

--- Selection Summary ---
Final counts per entity type in the golden set (based on predictions):
  - Natural Phenomenon: 51/50 examples
  - Organization: 62/50 examples
  - Ecosy

### Select Data Per Group

Selector based on multiset multicover problem:

In [5]:
def select_samples_with_msc_api(file_path, n_sample, entity_types_to_tag):
    """
    Selects annotated sentences using the correct multiset-multicover API.

    This function uses the GreedyCoverInstance object to find a near-optimal
    set of sentences meeting the coverage requirements for each entity type.

    Args:
        file_path (str): Path to the input JSON file.
        n_sample (int): The target number of examples for each entity type.
        entity_types_to_tag (list): A list of entity type strings to focus on.

    Returns:
        list: A list of the selected task dictionaries in their original format.
    """
    print("--- Starting Sample Selection using the multiset-multicover API ---")

    # 1. Load data and filter for already-annotated tasks
    try:
        with open(file_path, 'r') as f:
            all_tasks = json.load(f)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return []

    candidate_tasks = [
        task for task in all_tasks
        if task.get('annotations') and task['annotations'][0].get('result')
    ]
    print(f"Found {len(candidate_tasks)} sentences with existing annotations to select from.")

    # 2. Create the necessary mappings and initialize the solver instance
    # The library requires integer indices for elements, not strings.
    entity_to_idx = {entity: i for i, entity in enumerate(entity_types_to_tag)}
    num_entity_types = len(entity_types_to_tag)

    # Initialize the GreedyCoverInstance with the size of the element universe
    gci = mm.GreedyCoverInstance(num_entity_types)
    
    # This list will map the index of a multiset added to the solver
    # back to the original task data.
    task_map = []

    print("Preprocessing data and adding multisets to the solver instance...")
    for task in candidate_tasks:
        # Count occurrences of each target entity in this task's annotations
        entity_counts_in_task = Counter(
            ann['value']['labels'][0]
            for ann in task['annotations'][0]['result']
            if 'value' in ann and 'labels' in ann['value'] and ann['value']['labels'][0] in entity_types_to_tag
        )
        
        if entity_counts_in_task:
            # Convert the counted entities (strings) into the required format:
            # - A list of integer indices for the elements.
            # - A corresponding list of their multiplicities (counts).
            elements_in_task = []
            multiplicities_in_task = []
            for entity, count in entity_counts_in_task.items():
                elements_in_task.append(entity_to_idx[entity])
                multiplicities_in_task.append(count)

            # Add this task as a multiset to the solver instance
            gci.add_multiset(elements_in_task, multiplicity=multiplicities_in_task)
            
            # Keep track of the original task object at this index
            task_map.append(task)

    if not task_map:
        print("No tasks found containing the specified entity types.")
        return []

    # 3. Define the coverage requirements
    # A list where each entry corresponds to the coverage needed for an entity index.
    requirements = [n_sample] * num_entity_types
    print(f"\nSeeking to cover {num_entity_types} entity types with {n_sample} examples each.")

    # 4. Run the solver by calling the .cover() method
    print("Running the multiset-multicover solver...")
    selected_indices = gci.cover(requirements)
    print(f"Solver finished. Selected {len(selected_indices)} sentences.")

    # 5. Retrieve the selected tasks using the returned indices and the map
    selected_tasks = [task_map[i] for i in selected_indices]
    
    # 6. Print a final summary of the selection
    print("\n--- Selection Summary ---")
    final_counts = Counter()
    for task in selected_tasks:
        for ann in task['annotations'][0]['result']:
            try:
                label = ann['value']['labels'][0]
                if label in entity_types_to_tag:
                    final_counts[label] += 1
            except (KeyError, IndexError):
                continue

    print("Final counts per entity type:")
    for entity in entity_types_to_tag:
        count = final_counts.get(entity, 0)
        print(f"  - {entity}: {count}/{n_sample} examples")
    
    return selected_tasks

# --- EXAMPLE USAGE ---

# Define your parameters
FILE_PATH = "RESULTS/FT_ANNOTS/golden_50_EneSou_NatPhe_PhyArt_Org_TimPer_Sys_PhyPhe_BodPar_Org_Eco_MatExp_Dis_GeoFea_Che_Sat_Met_FieOfStu_BodOfWat_Pol_Ass_MetPhe_MeaDev_IntArt_NatDis_Qua_Per_Loc.json"
N_SAMPLES_PER_TYPE = 50
ENTITY_TYPES_TO_TAG = ["Asset", "Policy"]
# list(CLIRENER_LABELS_V1)
# ENTITY_TYPES_TO_TAG.remove("Other") # No optiomization for this class
# G1 (SMI & KB)
## ["Asset", "Policy"] 
## ["Method", "Field of Study", "Intellectual Artefact"]
# G2 (II & ???)
## ["Location", "Geographical Feature", "Body of Water", "Time Period", "Satellite"]
# G3 (MG & ???)
## ["Mathematical Expression", "Measuring Device", "Physical Phenomenon", "Quantity"]
# G4 (BS & MK)
## ["Body Part", "Chemical", "Disease", "Organism", "Ecosystem"]
# G5 (??? & ???)
## ["Energy Source", "Meteorological Phenomenon", "Natural Disaster", "Natural Phenomenon"]
# G6 (AP & ???)
## ["Physical Artefact", "Organization", "Person", "System"] 


# Run the new, API-correct selection function
selected_data = select_samples_with_msc_api(
    file_path=FILE_PATH,
    n_sample=N_SAMPLES_PER_TYPE,
    entity_types_to_tag=ENTITY_TYPES_TO_TAG
)

# Generate the filename from the entity types
filename_shorthand = create_shorthand_string(ENTITY_TYPES_TO_TAG)

# Remove annotations
selected_data = remove_annots(selected_data)

# Save the selected data to a new file
if selected_data:
    output_filename = f"RESULTS/FT_ANNOTS/mmp_{N_SAMPLES_PER_TYPE}_{filename_shorthand}.json"
    print(f"\nSaving {len(selected_data)} selected sentences to '{output_filename}'...")
    with open(output_filename, 'w') as f:
        json.dump(selected_data, f, indent=2)
    print("Done.")

--- Starting Sample Selection using the multiset-multicover API ---
Found 236 sentences with existing annotations to select from.
Preprocessing data and adding multisets to the solver instance...

Seeking to cover 2 entity types with 50 examples each.
Running the multiset-multicover solver...
Solver finished. Selected 36 sentences.

--- Selection Summary ---
Final counts per entity type:
  - Asset: 50/50 examples
  - Policy: 50/50 examples

Saving 36 selected sentences to 'RESULTS/FT_ANNOTS/mmp_50_Ass_Pol.json'...
Done.


One possible solution to mitigate author annotator bias is to use fnie-tuned predictions where predictions are all set to "Other" (while the correct spans are kept).

In [7]:
data_DIR = "RESULTS/FT_ANNOTS/mmp_50_Ass_Pol.json"
update_entity_labels(data_DIR, label_remap_rules_all2other)

Loading data from 'RESULTS/FT_ANNOTS/mmp_50_Ass_Pol.json'...
Successfully updated 319 label occurrences.
Saving updated data back to 'RESULTS/FT_ANNOTS/mmp_50_Ass_Pol.json'...
File saved successfully.


### Select Data Based on Previous Selection

We want to remain using same selected sentences, just asdd different model annotations: 

In [18]:
source_large = 'RESULTS/gliner_m_v2_5_CliReNER_v_1_1_28_SILVER_v1_261125_anotsready.json' # 26.11.2025.
# "RESULTS/gliner_medium_v21_clirener_2025_11_11_12_33_f873727e_anotsready.json" 
source_small = "RESULTS/annots_v2/mmp_50_PhyArt_Org_Per_Sys.json"
output_loc   = "RESULTS/annots_v2_ft_silver/mmp_50_PhyArt_Org_Per_Sys_FTS.json"

filter_json_data(source_large, source_small, output_loc)


Loaded 67 unique keys to filter by.
Filtering complete.
Source count: 14187
Filtered count: 67
Data contains previous annotations: False
Data has all labels set as "Other": True
Saved to: RESULTS/annots_v2_ft_silver/mmp_50_PhyArt_Org_Per_Sys_FTS.json


### Other datasets annotation

Load all datasets constructed with "Create multiset for all 27 classes (-"Other")"

In [14]:
OTHER_DATASETS_DIR = "RESULTS/OTHER_DATASETS_ANNOTATIONS/"

for file in os.listdir(OTHER_DATASETS_DIR):
    with open(OTHER_DATASETS_DIR+file, "r") as f:
        json_file = json.load(f)
        
    clean_json_file = set_all_labels_to_other(json_file)
    
    new_file_name = file.split(".")[0] + "_SPANS.json"
    with open(OTHER_DATASETS_DIR+new_file_name, "w") as f:
        json_file = json.dump(clean_json_file, f)

## Other

In [ ]:
for task in final_merged_data:
    pred_results = task.get("predictions")[0].get("result")
    task_data = task.get("data")
    for result in pred_results:
        if "Asset" in result["value"].get("labels"):
            print(task_data.get("paper_id"), task_data.get("sentence_id"), result["value"]["text"])           